# **🧠 Driver Drowsiness Detection using Eye Closure and Yawning Analysis with Deep Learning**

## 📌 Problem Statement

Driver fatigue is one of the major causes of road accidents worldwide. When drivers become drowsy, their alertness, reaction time, and decision-making ability significantly decrease, increasing the likelihood of dangerous situations on the road.

Traditional drowsiness detection systems often rely on vehicle behavior analysis (such as steering patterns or lane deviation) or wearable sensors that monitor physiological signals. However, these approaches can be intrusive, inconvenient for drivers, or unreliable in real-world driving conditions.

To address these limitations, there is a need for a vision-based, non-intrusive driver monitoring system that can detect early signs of fatigue. By analyzing facial features such as eye closure and yawning patterns, a deep learning model can automatically identify whether a driver is alert or drowsy and help prevent accidents through early warnings.

## 🎯 Objective

The main objectives of this project are:

- To develop a deep learning-based driver drowsiness detection system using facial image data.

- To detect early signs of fatigue such as eye closure and yawning from driver images.

- To preprocess and train the model using a dataset of approximately 3000 images.

- To build a Convolutional Neural Network (CNN) model capable of classifying driver states such as alert and drowsy.

- To evaluate the model performance using metrics such as accuracy, precision, recall, and confusion matrix.

- To design a system that can be extended for real-time driver monitoring using webcam input.

## 📂 Dataset Collection

The dataset used in this project consists of real human facial images representing different driver conditions related to fatigue and alertness. The images are categorized based on eye state and yawning behavior, which are key physiological indicators of driver drowsiness.

The dataset contains approximately 4,800 images, which are manually organized into the following four classes:

- **Eyes Open** – Images where the driver's eyes are open, representing an **alert driving state**.

- **Eyes Closed** – Images where the driver's eyes are closed, indicating a **possible drowsy state**.

- **Yawn** – Images where the driver is yawning, which is a common sign of **fatigue and sleepiness**.

- **No Yawn** – Images where the driver is not yawning, representing a **normal and attentive state**.

These images are used to train and evaluate a deep learning model that can automatically detect signs of driver fatigue by analyzing facial features.

The dataset is divided into training and testing sets to ensure proper model evaluation and generalization.

## **Step 1: Dataset Loading**

#### 🧾 Import Required Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

**🔍 What this code does**

- os → Helps access folders and files in the dataset directory.

- numpy → Used for numerical operations and storing image arrays.

- matplotlib → Used to visualize images.

- PIL (Python Imaging Library) → Used to load and process images.

**💡 Insight**

Importing these libraries provides the necessary tools for reading, processing, and visualizing image data, which is essential before building a deep learning model.

#### 📂 Define Dataset Path

In [ ]:
dataset_path = "dataset"
classes = ["Closed", "no_yawn", "Open", "yawn"]

**🔍 What this code does**

- Defines the main dataset folder location.

- Lists the four image categories present in the dataset.

**💡 Insight**

Organizing data into separate class folders helps the model learn patterns for each driver state, which improves classification accuracy.

#### 📊 Count Images in Each Class

In [ ]:
for category in classes:
    path = os.path.join(dataset_path, category)
    num_images = len(os.listdir(path))
    print(f"{category}: {num_images} images")

**🔍 What this code does**

- Accesses each folder inside the dataset.

- Counts the number of images in each class.

- Prints the distribution of images.

**💡 Insight**

Checking the number of images in each class helps us understand whether the dataset is balanced or imbalanced. Balanced datasets help the model learn fairly from all categories.

#### 🖼️ Display Sample Images

In [ ]:
plt.figure(figsize=(10,6))

for i, category in enumerate(classes):
    path = os.path.join(dataset_path, category)
    img_name = os.listdir(path)[0]
    img_path = os.path.join(path, img_name)

    img = Image.open(img_path)

    plt.subplot(2,2,i+1)
    plt.imshow(img)
    plt.title(category)
    plt.axis("off")

plt.show()

**🔍 What this code does**

- Selects one sample image from each class.

- Displays the images using Matplotlib.

**💡 Insight**

- Visualizing sample images helps us:

- Verify that the dataset is correctly labeled

- Understand visual differences between classes

- Ensure the images are suitable for training the deep learning model

## **📂 Step 2: Dataset Organization**

**📌 Introduction**

To train and evaluate a deep learning model effectively, the dataset must be divided into three subsets:

- Training Set – Used to train the model and learn patterns from the images.

- Validation Set – Used to tune model parameters and prevent overfitting.

- Test Set – Used to evaluate the final performance of the trained model on unseen data.

Splitting the dataset ensures that the model can generalize well to new data rather than simply memorizing the training images.

In this project, the dataset will be divided as follows:

- 70% Training Data

- 15% Validation Data

- 15% Test Data

In [ ]:
import shutil
import random

#### 📁 Create Train, Validation, and Test Folders

In [ ]:
dataset_path = "dataset"
output_path = "dataset_split"

classes = ["Closed", "no_yawn", "Open", "yawn"]

🔍 What this code does

- Defines the original dataset folder

- Creates a new folder for the split dataset

- Lists all class labels

In [ ]:
splits = ["train", "val", "test"]

for split in splits:
    for category in classes:
        path = os.path.join(output_path, split, category)
        os.makedirs(path, exist_ok=True)

💡 Insight

Organizing the dataset this way allows deep learning frameworks to easily load training and validation data during model training.

In [ ]:
for category in classes:
    
    src = os.path.join(dataset_path, category)
    images = os.listdir(src)
    
    random.shuffle(images)
    
    train_size = int(0.7 * len(images))
    val_size = int(0.15 * len(images))
    
    train_images = images[:train_size]
    val_images = images[train_size:train_size + val_size]
    test_images = images[train_size + val_size:]
    
    for img in train_images:
        shutil.copy(os.path.join(src, img),
                    os.path.join(output_path, "train", category))
    
    for img in val_images:
        shutil.copy(os.path.join(src, img),
                    os.path.join(output_path, "val", category))
    
    for img in test_images:
        shutil.copy(os.path.join(src, img),
                    os.path.join(output_path, "test", category))

**🔍 What This Code Does**

- Reads all images from each class folder

- Randomly shuffles them

- Splits them into:

- 70% Training

- 15% Validation

- 15% Test

- Copies images into the newly created folders.

**💡 Insight**

Dataset splitting is crucial because it allows the model to be trained, validated, and tested on separate data. This helps evaluate whether the model can generalize to unseen images, which is essential for real-world applications like driver drowsiness detection.

## **🔧 Step 3: Data Preprocessing**

**📌 Introduction**

Before training the deep learning model, the images must be preprocessed to ensure consistency and improve model performance. Image preprocessing involves resizing, normalization, and data augmentation.

These steps help the model learn better patterns and prevent overfitting.

In this step we perform:

- Image resizing to 224 × 224

- Pixel normalization

- Data augmentation techniques:

        Rotation

        Zoom

        Brightness variation

        Horizontal flipping

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

🔍 What this code does

This imports ImageDataGenerator, a utility from TensorFlow/Keras used for:

- Loading images from folders

- Performing image preprocessing

- Applying data augmentation

💡 Insight

Using ImageDataGenerator allows preprocessing and augmentation to happen automatically during training, which improves efficiency and prevents the need to store multiple augmented images.

#### ⚙️ Define Image Size and Batch Size

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

#### 🔄 Data Augmentation for Training Data

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True
)

🔍 What this code does

This applies data augmentation techniques:

| Parameter              | Function                                          |
| ---------------------- | ------------------------------------------------- |
| `rescale=1./255`       | Normalizes pixel values from **0–255 → 0–1**      |
| `rotation_range=20`    | Randomly rotates images up to **20 degrees**      |
| `zoom_range=0.2`       | Randomly zooms images by **20%**                  |
| `brightness_range`     | Adjusts brightness to simulate different lighting |
| `horizontal_flip=True` | Flips images horizontally                         |

💡 Insight

Data augmentation increases the diversity of training data, helping the model become more robust to variations such as different head positions, lighting conditions, and camera angles.

#### 📂 Validation and Test Data Generator

In [ ]:
val_test_datagen = ImageDataGenerator(rescale=1./255)

🔍 What this code does

Only normalizes pixel values

Does not apply augmentation

💡 Insight

Validation and test data should represent real-world conditions. Applying augmentation here could distort the true performance evaluation of the model.

📥 Load Training Dataset

In [ ]:
train_generator = train_datagen.flow_from_directory(
    "dataset_split/train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

🔍 What this code does

Loads training images from the train folder

Resizes them to 224 × 224

Applies data augmentation

Converts class labels into categorical format

💡 Insight

This method allows the model to automatically read images from folders and assign labels, simplifying dataset handling

📥 Load Validation Dataset

In [ ]:
val_generator = val_test_datagen.flow_from_directory(
    "dataset_split/val",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

🔍 What this code does

Loads validation images for model tuning and monitoring training performance.

💡 Insight

Validation data helps detect overfitting, ensuring the model performs well on unseen images.

📥 Load Test Dataset

In [ ]:
test_generator = val_test_datagen.flow_from_directory(
    "dataset_split/test",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

🔍 What this code does

Loads the test dataset, which will be used for final model evaluation.

💡 Insight

The test set represents completely unseen data, allowing an unbiased assessment of model performance.

## **🧠 Step 4: Model Selection**

📌 Introduction

In this step, two different deep learning approaches are used to detect driver fatigue from facial images.

1. Custom Convolutional Neural Network (CNN)

   A CNN model built from scratch to learn image features such as eye closure and yawning patterns directly from the dataset.

2. Transfer Learning using MobileNetV2

   A pretrained deep learning model trained on the ImageNet dataset. The model already knows many general visual features, so we reuse those features and fine-tune it for our driver drowsiness dataset.

Both models are trained and evaluated to compare:

- Accuracy

- Training time

- Generalization performance

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D

🔍 What this code does

This imports the required deep learning components from TensorFlow.

| Library      | Purpose                                             |
| ------------ | --------------------------------------------------- |
| tensorflow   | Main deep learning framework                        |
| Sequential   | Used to build CNN layers sequentially               |
| Conv2D       | Convolution layer to extract image features         |
| MaxPooling2D | Reduces image size while keeping important features |
| Dense        | Fully connected neural network layer                |
| Dropout      | Prevents overfitting                                |
| MobileNetV2  | Pretrained deep learning model                      |

💡 Insight

Importing these modules provides the necessary tools to build both custom CNN models and transfer learning architectures.

### 🧠 Model 1: Custom CNN Architecture

In [ ]:
cnn_model = Sequential([
    
    Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),

    Dense(128, activation='relu'),
    Dropout(0.5),

    Dense(4, activation='softmax')
])

🔍 What this code does

This creates a CNN model from scratch.

Layer explanation:

| Layer      | Purpose                                    |
| ---------- | ------------------------------------------ |
| Conv2D     | Detects edges, shapes, and facial features |
| MaxPooling | Reduces image size and computation         |
| Flatten    | Converts feature maps into a vector        |
| Dense      | Performs classification                    |
| Dropout    | Reduces overfitting                        |

Output layer: Dense(4, activation='softmax')

💡 Insight

A custom CNN learns dataset-specific features, which is useful when the dataset closely represents the target problem.

However, training from scratch requires more data and training time.

##### ⚙️ Compile CNN Model

In [ ]:
cnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

🔍 What this code does

| Parameter                | Purpose                                      |
| ------------------------ | -------------------------------------------- |
| adam                     | Optimizer used to update model weights       |
| categorical_crossentropy | Loss function for multi-class classification |
| accuracy                 | Evaluation metric                            |

💡 Insight

The Adam optimizer is widely used in deep learning because it adapts learning rates automatically during training.

### 🚀 Model 2: Transfer Learning with MobileNetV2

In [ ]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

🔍 What this code does

Loads MobileNetV2 pretrained model without its final classification layer.

| Parameter          | Meaning                     |
| ------------------ | --------------------------- |
| weights="imagenet" | Uses pretrained weights     |
| include_top=False  | Removes original classifier |
| input_shape        | Input image size            |

💡 Insight

Transfer learning allows the model to reuse features learned from millions of images in ImageNet, improving performance even with smaller datasets.

##### 🧠 Add Custom Classification Layers

In [ ]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)

output = Dense(4, activation='softmax')(x)

mobilenet_model = Model(inputs=base_model.input, outputs=output)

🔍 What this code does

Adds custom layers on top of MobileNetV2:

1. GlobalAveragePooling

2. Dense layer

3. Dropout

4. Output classification layer

💡 Insight

These layers adapt the pretrained model to your driver drowsiness dataset.

##### ⚙️ Compile MobileNet Model

In [ ]:
mobilenet_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

🔍 What this code does

Compiles the MobileNet transfer learning model with the same optimizer and loss function as the CNN.

💡 Insight

Using the same configuration allows a fair comparison between the two models.

### 📊 Model Summary

In [ ]:
cnn_model.summary()
mobilenet_model.summary()

🔍 What this code does

Displays:

- Number of layers

- Parameters

- Output shapes

💡 Insight

The summary helps analyze model complexity and computational cost.

### 📈 Expected Comparison

| Metric           | Custom CNN | MobileNetV2 |
| ---------------- | ---------- | ----------- |
| Training Time    | Higher     | Lower       |
| Accuracy         | Moderate   | Higher      |
| Data Requirement | More       | Less        |
| Generalization   | Moderate   | Strong      |

💡 Final Insight

Transfer learning models such as MobileNetV2 architecture typically outperform custom CNN models when datasets are moderate in size, because pretrained features improve learning efficiency.

## 🧠 Step 5: Model Development

📌 Introduction

In this step, two separate classification models are developed to detect driver fatigue indicators from facial images.

1. Eye State Classification Model

    This model predicts whether the driver’s eyes are open or closed, which is an important indicator of drowsiness.

2. Mouth State Classification Model

    This model predicts whether the driver is yawning or not yawning, which can indicate fatigue.

Both models use transfer learning with the pretrained architecture MobileNetV2 architecture to leverage features learned from the ImageNet dataset. Custom classification layers are added on top to adapt the model to the driver drowsiness dataset.

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model

🔍 What this code does

- Imports deep learning tools from TensorFlow.

- Loads MobileNetV2 for transfer learning.

- Provides layers for building classification models.

💡 Insight

Using pretrained models helps extract general visual features such as edges, shapes, and textures, improving model performance even when the dataset is relatively small.

### 👁️ Eye State Classification Model (Open vs Closed)

In [ ]:
base_model_eye = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model_eye.trainable = False

🔍 What this code does

- Loads MobileNetV2 without its original classification layer.

- Freezes pretrained weights so the network acts as a feature extractor.

💡 Insight

Freezing the base model prevents previously learned features from being modified, allowing the network to focus on learning the new classification task.

### ➕ Add Custom Classification Layers

In [ ]:
x = base_model_eye.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.5)(x)

eye_output = Dense(2, activation="softmax")(x)

eye_model = Model(inputs=base_model_eye.input, outputs=eye_output)

🔍 What this code does

Adds new layers on top of the pretrained network:

| Layer                | Purpose                                |
| -------------------- | -------------------------------------- |
| GlobalAveragePooling | Converts feature maps to a vector      |
| Dense                | Learns dataset-specific features       |
| Dropout              | Prevents overfitting                   |
| Softmax              | Predicts probabilities for two classes |

Output classes:

    Open

    Closed

💡 Insight

Custom layers allow the pretrained model to adapt specifically to eye state detection, which is critical for identifying driver drowsiness.

### ⚙️ Compile Eye State Model

In [ ]:
eye_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

🔍 What this code does

Configures the model for training.

| Component          | Role                          |
| ------------------ | ----------------------------- |
| Adam optimizer     | Adjusts model weights         |
| Cross-entropy loss | Measures classification error |
| Accuracy metric    | Evaluates model performance   |

💡 Insight

The Adam optimizer is widely used because it efficiently handles large datasets and complex neural networks.

### 😮 Mouth State Classification Model (Yawn vs No Yawn)

In [ ]:
base_model_mouth = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model_mouth.trainable = False

🔍 What this code does

Loads another MobileNetV2 network for detecting mouth states.

💡 Insight

Separating eye and mouth detection allows each model to specialize in different facial features, improving overall fatigue detection accuracy.

#### ➕ Add Custom Layers for Mouth Detection

In [ ]:
y = base_model_mouth.output
y = GlobalAveragePooling2D()(y)
y = Dense(128, activation="relu")(y)
y = Dropout(0.5)(y)

mouth_output = Dense(2, activation="softmax")(y)

mouth_model = Model(inputs=base_model_mouth.input, outputs=mouth_output)

🔍 What this code does

Adds custom classification layers for mouth-state prediction.

Output classes:

        Yawn
        No_Yawn

💡 Insight

This model focuses on detecting mouth opening patterns, which can signal fatigue and reduced alertness.

### ⚙️ Compile Mouth State Model

In [ ]:
mouth_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

🔍 What this code does

Prepares the mouth detection model for training.

💡 Insight

Using the same optimizer and loss function ensures consistent training behavior across both models, making their performance easier to compare.

#### 📊 Model Summary

In [ ]:
eye_model.summary()
mouth_model.summary()

📌 Summary of Model Development

At this stage:

- Eye State Model detects

    Open eyes

    Closed eyes

- Mouth State Model detects

    Yawning

    No yawning

Both models use transfer learning with MobileNetV2 architecture and custom classification layers to adapt the pretrained network to the driver fatigue detection problem.

## 🧠 Step 6: Model Training

📌 Goal

In this step we train the model by:

- Freezing base layers of MobileNetV2

- Training only the custom classification head

- Using Adam optimizer

- Monitoring training and validation accuracy/loss



In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

What this code does

Imports all deep learning tools needed for:

- loading images

- building the CNN model

- training and visualization

2️⃣ Create Image Data Generators

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

test_datagen = ImageDataGenerator(rescale=1./255)

| Parameter       | Purpose                         |
| --------------- | ------------------------------- |
| rescale         | Normalize pixel values          |
| rotation        | Helps model learn rotated faces |
| zoom            | Improves generalization         |
| horizontal_flip | Handles different face angles   |


Insight

This process is called data augmentation.
It helps the model learn better general visual patterns.

#### 3️⃣ Load Training Dataset

In [ ]:
train_data = train_datagen.flow_from_directory(
    "dataset_split/train",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical"
)

#### 4️⃣ Load Validation Dataset

In [ ]:
val_data = val_datagen.flow_from_directory(
    "dataset_split/val",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical"
)

Why validation dataset is used

Validation helps monitor:

- validation accuracy

- validation loss

during training.

#### 5️⃣ Load Test Dataset

In [ ]:
test_data = test_datagen.flow_from_directory(
    "dataset_split/test",
    target_size=(224,224),
    batch_size=16,
    class_mode="categorical",
    shuffle=False
)

#### 6️⃣ Load Pretrained MobileNetV2

In [ ]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

What this does

Loads pretrained weights from the ImageNet dataset.

The model already knows how to detect:

- edges

- textures

- shapes

- objects

#### 7️⃣ Freeze Base Layers

In [ ]:
for layer in base_model.layers:
    layer.trainable = False

Why freeze layers?

We freeze the base layers so that:

- pretrained features remain unchanged

- training becomes faster

- fewer parameters are trained

Only the new classification layers will learn.

#### 8️⃣ Add Custom Classification Head

In [ ]:
x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(128, activation="relu")(x)
x = Dropout(0.5)(x)

output = Dense(4, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

Because we have 4 classes:

- Closed
- Open
- no_yawn
- yawn

Softmax converts outputs into class probabilities.

#### 9️⃣ Compile the Model

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

| Component                | Purpose                             |
| ------------------------ | ----------------------------------- |
| Adam                     | Fast and adaptive optimizer         |
| categorical_crossentropy | Best for multi-class classification |
| accuracy                 | Measures correct predictions        |


#### 🔟 Train the Model

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=30
)

#### 11️⃣ Monitor Validation Accuracy and Loss

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend(["Train","Validation"])
plt.show()

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])

plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(["Train","Validation"])
plt.show()

What These Graphs Tell Us

Good Model

        Train accuracy ↑
        Validation accuracy ↑
        Loss ↓

Overfitting

        Train accuracy ↑
        Validation accuracy ↓
        Underfitting
Both accuracies low

Monitoring validation metrics ensures the model generalizes well.

## Step 8: Model Evaluation

1️⃣ Import Required Libraries

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

2️⃣ Evaluate Model on Test Dataset

In [ ]:
test_loss, test_accuracy = model.evaluate(test_data)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

In [ ]:
test_loss, test_accuracy = model.evaluate(val_data)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

In [ ]:
test_loss, test_accuracy = model.evaluate(train_data)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

What this code does

Runs the trained model on the unseen test dataset.

Output example:

Test Loss: 0.28
Test Accuracy: 0.92

Meaning:

92% of the test images were classified correctly.

Formula:

Accuracy = Correct Predictions / Total Predictions

However, accuracy alone is not enough, especially if classes are imbalanced.

That is why we use confusion matrix, precision, and recall.

3️⃣ Generate Predictions on Test Data

In [ ]:
predictions = model.predict(test_data)

y_pred = np.argmax(predictions, axis=1)
y_true = test_data.classes
y_true,y_pred

What this code does

Neural networks output probability values.

Example output:

[0.05, 0.90, 0.03, 0.02]

This means:

Closed  = 5%
Open    = 90%
no_yawn = 3%
yawn    = 2%

Using:

np.argmax()

we select the highest probability class.

4️⃣ Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

print("Confusion Matrix:")
print(cm)

Interpretation:

45 Closed images correctly predicted.

3 Closed images misclassified as Open.

This helps identify where the model makes mistakes.

5️⃣ Visualize Confusion Matrix

In [ ]:
plt.figure(figsize=(6,5))

sns.heatmap(cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=test_data.class_indices.keys(),
            yticklabels=test_data.class_indices.keys())

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

What this code does

Creates a heatmap visualization of the confusion matrix.

This makes it easier to see:

- correct predictions

- misclassifications

6️⃣ Precision, Recall and F1 Score

In [ ]:
report = classification_report(
    y_true,
    y_pred,
    target_names=test_data.class_indices.keys()
)

print(report)

Why These Metrics Matter for Driver Drowsiness

For fatigue detection systems:

Recall is very important.

If the model fails to detect yawning or closed eyes, the driver may remain undetected while drowsy.

That is why evaluation must include:

- accuracy

- precision

- recall

- confusion matrix

## Step 8: Decision Fusion Logic (3-Level Fatigue Classification)

Concept

We map the 4 model outputs → 3 fatigue stages.

| Model Prediction | Fatigue Level      | Meaning              |
| ---------------- | ------------------ | -------------------- |
| Open             | Alert (0)          | Driver normal        |
| no_yawn          | Alert (0)          | Driver normal        |
| yawn             | Mild Fatigue (1)   | Driver getting tired |
| Closed           | Severe Fatigue (2) | Driver drowsy        |


1️⃣ Create Class Label Mapping

In [ ]:
print(train_data.class_indices)

In [ ]:
class_labels = {v:k for k,v in train_data.class_indices.items()}
print(class_labels)

2️⃣ Define Fatigue Decision Logic

In [ ]:
def fatigue_stage(predicted_class):

    if predicted_class in ["Open","no_yawn"]:
        return 0, "Alert"

    elif predicted_class == "yawn":
        return 1, "Mild Fatigue"

    elif predicted_class == "Closed":
        return 2, "Severe Fatigue"

What this code does

Implements the 3-level fatigue classification system.

3️⃣ Predict Driver Image

In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np

img_path = "no yawn.jfif"

img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img)

img_array = img_array / 255.0
img_array = np.expand_dims(img_array, axis=0)

What this code does

Steps:

- Load image

- Resize to model input size

- Convert to array

- Normalize pixel values

- Add batch dimension

This prepares the image for model prediction.

4️⃣ Get Model Prediction

In [ ]:
prediction = model.predict(img_array)

predicted_index = np.argmax(prediction)

predicted_label = class_labels[predicted_index]

print("Model Prediction:", predicted_label)

What this code does

1. Model predicts probabilities

2. argmax() selects the highest probability class

3. Converts numeric label to class name



5️⃣ Apply Decision Fusion Logic

In [ ]:
stage_id, stage_name = fatigue_stage(predicted_label)

print("Fatigue Level:", stage_id)
print("Driver State:", stage_name)

In [ ]:
class_labels = {0:'Closed',1:'Open',2:'no_yawn',3:'yawn'}

def fatigue_stage(predicted_class):

    if predicted_class in ["Open","no_yawn"]:
        return 0, "Alert"

    elif predicted_class == "yawn":
        return 1, "Mild Fatigue"

    elif predicted_class == "Closed":
        return 2, "Severe Fatigue"


prediction = model.predict(img_array)

predicted_index = np.argmax(prediction)

predicted_label = class_labels[predicted_index]

stage_id, stage_name = fatigue_stage(predicted_label)

print("Model Prediction:", predicted_label)
print("Fatigue Level:", stage_id)
print("Driver State:", stage_name)

Final System Output

| CNN Output | Final Driver State |
| ---------- | ------------------ |
| Open       | Alert              |
| no_yawn    | Alert              |
| yawn       | Mild Fatigue       |
| Closed     | Severe Fatigue     |


## Step 9: Driver Fatigue Progression Curve

1️⃣ Import Required Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image

2️⃣ Define Class Labels

In [ ]:
class_labels = {
    0: "Closed",
    1: "Open",
    2: "no_yawn",
    3: "yawn"
}

3️⃣ Create Fatigue Mapping Function

In [ ]:
def fatigue_stage(predicted_label):

    if predicted_label in ["Open", "no_yawn"]:
        return 0

    elif predicted_label == "yawn":
        return 1

    elif predicted_label == "Closed":
        return 2

Insight

This converts facial state detection into driver fatigue level detection.

4️⃣ Simulate Continuous Driving Session

In [ ]:
session_folder = "dataset_split/test"

fatigue_levels = []

What this does

Creates a list to store fatigue level for each frame.

5️⃣ Process Images Sequentially

In [ ]:
for class_folder in os.listdir(session_folder):

    folder_path = os.path.join(session_folder, class_folder)

    for img_name in os.listdir(folder_path):

        img_path = os.path.join(folder_path, img_name)

        img = image.load_img(img_path, target_size=(128,128))
        img_array = image.img_to_array(img)

        img_array = img_array / 255.0
        img_array = np.expand_dims(img_array, axis=0)

        prediction = model.predict(img_array)

        predicted_index = np.argmax(prediction)

        predicted_label = class_labels[predicted_index]

        stage = fatigue_stage(predicted_label)

        fatigue_levels.append(stage)

What this code does

For every image:

1. Load the image

2. Preprocess it

3. Predict using the trained CNN

4. Convert prediction → fatigue stage

5. Store stage value

6️⃣ Create Time Intervals

In [ ]:
frames_per_minute = 10

fatigue_over_time = []

for i in range(0, len(fatigue_levels), frames_per_minute):

    interval = fatigue_levels[i:i+frames_per_minute]

    avg_fatigue = np.mean(interval)

    fatigue_over_time.append(avg_fatigue)

What this code does

Groups predictions into time intervals.

7️⃣ Plot Fatigue Progression Curve

In [ ]:
time_axis = list(range(len(fatigue_over_time)))

plt.figure(figsize=(8,5))

plt.plot(time_axis, fatigue_over_time, marker='o')

plt.title("Driver Fatigue Progression Curve")
plt.xlabel("Time Interval (minutes)")
plt.ylabel("Fatigue Level")

plt.yticks([0,1,2], ["Alert","Mild Fatigue","Severe Fatigue"])

plt.grid(True)

plt.show()

Identify Transition Points

In [ ]:
for i in range(1, len(fatigue_over_time)):

    if fatigue_over_time[i] > fatigue_over_time[i-1]:

        print("Fatigue increased at minute:", i)

## Step 10: Performance Analysis

1️⃣ Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

2️⃣ Generate Predictions on Test Dataset

In [ ]:
predictions = model.predict(test_data)

y_pred = np.argmax(predictions, axis=1)
y_true = test_data.classes

3️⃣ Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

print("Confusion Matrix")
print(cm)

4️⃣ Visualize Confusion Matrix

In [ ]:
plt.figure(figsize=(6,5))

sns.heatmap(cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=test_data.class_indices.keys(),
            yticklabels=test_data.class_indices.keys())

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

5️⃣ Precision, Recall and F1-score

In [ ]:
report = classification_report(
    y_true,
    y_pred,
    target_names=test_data.class_indices.keys()
)

print(report)

6️⃣ Compare Performance Across Classes

In [ ]:
classes = list(test_data.class_indices.keys())

precision = []
recall = []

report_dict = classification_report(
    y_true,
    y_pred,
    target_names=classes,
    output_dict=True
)

for c in classes:
    precision.append(report_dict[c]['precision'])
    recall.append(report_dict[c]['recall'])

plt.figure(figsize=(8,5))

x = np.arange(len(classes))

plt.bar(x-0.2, precision, width=0.4, label="Precision")
plt.bar(x+0.2, recall, width=0.4, label="Recall")

plt.xticks(x, classes)
plt.title("Class-wise Performance")
plt.legend()

plt.show()

7️⃣ Identify Error Cases

In [ ]:
errors = np.where(y_pred != y_true)[0]

print("Number of errors:", len(errors))

In [ ]:
for i in errors[:5]:

    img_path = test_data.filepaths[i]

    print("Actual:", y_true[i])
    print("Predicted:", y_pred[i])
    print("Image:", img_path)

## Final Conclusion

In this project, a deep learning–based driver fatigue detection system was developed to automatically classify eye and mouth states related to drowsiness. A convolutional neural network using **MobileNetV2** was trained to recognize four classes: **Closed**, **Open**, **no_yawn**, and **yawn**. The dataset was preprocessed using image resizing, normalization, and data augmentation to improve the model’s generalization capability.

After training and evaluation, the model achieved a **test accuracy of 91%**, indicating strong performance in distinguishing between different driver states. Performance analysis using confusion matrices and class-wise metrics showed that the model performs consistently across most classes, with only minor misclassifications occurring between visually similar states such as *yawn* and *no_yawn*.

This model can be applied in **real-world driver monitoring systems**. By integrating the trained model with a camera and real-time image processing libraries such as OpenCV, the system can continuously monitor a driver’s face and detect signs of fatigue. If prolonged eye closure or frequent yawning is detected, the system can trigger alerts such as alarms or warnings to help prevent accidents caused by drowsy driving.

Overall, the developed model demonstrates that deep learning techniques can effectively detect driver fatigue and has potential applications in **smart vehicles, transportation safety systems, and advanced driver assistance systems (ADAS)**. Future improvements could include larger and more diverse datasets, real-time video analysis, and integration with additional physiological indicators to further increase reliability and accuracy.


In [ ]:
# Save trained model
model.save("model.h5")

print("Model saved successfully")